In [1]:
import weather
import subprocess
import json
import datetime
import pandas as pd

In [2]:
info = weather.weatherInfo("BS1 1NR")

In [3]:
info.getArt()
info.getDoI()

(                        time  temperature_2m  apparent_temperature  \
 0  2026-03-28 00:00:00+00:00             8.0                   4.7   
 1  2026-03-28 00:15:00+00:00             7.8                   4.5   
 2  2026-03-28 00:30:00+00:00             7.7                   4.5   
 3  2026-03-28 00:45:00+00:00             7.5                   4.5   
 4  2026-03-28 01:00:00+00:00             7.3                   4.3   
 ..                       ...             ...                   ...   
 93 2026-03-28 23:15:00+00:00             5.4                   1.9   
 94 2026-03-28 23:30:00+00:00             5.3                   1.9   
 95 2026-03-28 23:45:00+00:00             5.3                   2.0   
 96 2026-03-29 00:00:00+00:00             5.2                   2.0   
 97 2026-03-29 00:15:00+00:00             5.3                   2.2   
 
     precipitation  precipitation_probability  
 0             0.0                          0  
 1             0.0                          0  
 2

In [4]:
details_forcast     = subprocess.getoutput(f'curl -fGsS "https://api.open-meteo.com/v1/forecast?latitude={info.gps_coords[0]}&longitude={info.gps_coords[1]}&minutely_15=temperature_2m,apparent_temperature,precipitation,precipitation_probability&forecast_days=2&timezone=auto"')
details_air_quality = subprocess.getoutput(f'curl -fGsS "https://air-quality-api.open-meteo.com/v1/air-quality?latitude={info.gps_coords[0]}&longitude={info.gps_coords[1]}&hourly=birch_pollen,european_aqi&forecast_days=2&timezone=auto"')
data_forcast = json.loads(details_forcast)
data_air_quality = json.loads(details_air_quality)

In [5]:
df_air_quality = pd.DataFrame({
    "time": [info.tz.localize(datetime.datetime.fromisoformat(t)) for t in data_air_quality["hourly"]["time"]],
    "birch_pollen": data_air_quality["hourly"]["birch_pollen"],
    "european_aqi": data_air_quality["hourly"]["european_aqi"]
})
df_air_quality  = df_air_quality.set_index("time").resample(datetime.timedelta(minutes=5)).interpolate().reset_index()

ValueError: cannot reindex on an axis with duplicate labels